# Knowledge Base Ingestion (Gemini + BM25)
This notebook ingests the corpus into a Qdrant Cloud instance using:
- **Dense Embeddings**: Google `gemini-embedding-001`
- **Sparse Vectors**: Qdrant's internal `BM25`

**Target Instance**: `https://qdrant-146934558634.asia-southeast1.run.app`

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pathlib import Path
from qdrant_client import QdrantClient, models
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Load environment variables
load_dotenv()

# Configuration
class Config:
    QDRANT_URL = "https://qdrant-146934558634.asia-southeast1.run.app"
    QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
    COLLECTION_NAME = "knowledgebase_gemini_bm25"
    
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    GEMINI_MODEL = "models/gemini-embedding-001"
    
    CORPUS_PATH = Path("data/corpus.parquet")

if not Config.QDRANT_API_KEY:
    print("⚠️ WARNING: QDRANT_API_KEY not found. Ensure it is set in your .env file.")
else:
    print("Configuration loaded.")

In [ ]:
# Initialize Client
client = QdrantClient(url=Config.QDRANT_URL, api_key=Config.QDRANT_API_KEY)

# Initialize Gemini Embeddings
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model=Config.GEMINI_MODEL,
    google_api_key=Config.GOOGLE_API_KEY
)

print("Client and Embeddings initialized.")

In [ ]:
# Load Corpus
if Config.CORPUS_PATH.exists():
    corpus_df = pd.read_parquet(Config.CORPUS_PATH)
    print(f"Loaded {len(corpus_df)} documents from {Config.CORPUS_PATH}")
else:
    raise FileNotFoundError(f"Corpus file not found: {Config.CORPUS_PATH}")

In [ ]:
def ingest_data():
    # 1. Check Embedding Dimension dynamically
    print("Checking embedding dimension...")
    try:
        sample_vec = gemini_embeddings.embed_query("test")
        gemini_dim = len(sample_vec)
        print(f"Gemini Dimension: {gemini_dim}")
    except Exception as e:
        print(f"Error getting embedding dimension: {e}")
        return

    # 2. Recreate Collection
    print(f"Recreating collection '{Config.COLLECTION_NAME}'...")
    client.recreate_collection(
        collection_name=Config.COLLECTION_NAME,
        vectors_config={
            "gemini": models.VectorParams(size=gemini_dim, distance=models.Distance.COSINE)
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
        }
    )
    
    # 3. Create Payload Indexes
    print("Creating payload indexes...")
    fields = ["metadata.plant", "metadata.disease", "metadata.type", "metadata.source"]
    for field in fields:
        client.create_payload_index(
            collection_name=Config.COLLECTION_NAME,
            field_name=field,
            field_schema=models.PayloadSchemaType.KEYWORD
        )
    
    # 4. Ingest
    points = []
    batch_size = 50
    docs = corpus_df.to_dict('records')
    
    print(f"Starting ingestion of {len(docs)} documents...")
    for doc in tqdm(docs):
        content = doc['contents']
        doc_id = doc['doc_id']
        metadata = doc['metadata'] if doc['metadata'] else {}
        
        # Generate Embeddings
        try:
            gemini_vec = gemini_embeddings.embed_query(content)
            
            point = models.PointStruct(
                id=doc_id,
                vector={
                    "gemini": gemini_vec,
                    "bm25": models.Document(text=content, model="Qdrant/bm25")
                },
                payload={
                    "page_content": content,
                    "metadata": metadata
                }
            )
            points.append(point)
        except Exception as e:
            print(f"Error processing doc {doc_id}: {e}")
            continue

        if len(points) >= batch_size:
            client.upsert(collection_name=Config.COLLECTION_NAME, points=points)
            points = []
            
    if points:
        client.upsert(collection_name=Config.COLLECTION_NAME, points=points)
        
    print("Ingestion complete!")

# Run Ingestion
ingest_data()